# **Notebook 03: RAG Generation and Signal Extraction**

**Description:** This notebook prepares and optionally executes resumable OpenAI Batch requests for direct, Hybrid RRF, and reranked RAG with gpt-5.6-luna, then separates provider availability from answer correctness, task success, abstention, usage, and cost.

**Main Questions This Notebook Should Answer:**

- What is the complete request plan and conservative spending reservation?
- When does retrieval improve answer quality relative to direct generation?
- Which requests failed or remained unavailable?
- Which examples have complete paired results for policy analysis?

---

# **Part 1: Setup, Policy Guardrails, and Artifact Contract**

### **Environment Setup**


In [1]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / 'src/adaptive_rag').is_dir() and (p / 'config').is_dir())
sys.path.insert(0, str(REPO_ROOT / 'src'))
ROOT = Path(os.environ.get('ADAPTIVE_RAG_PROJECT_ROOT', REPO_ROOT)).resolve()
from adaptive_rag.luna_v2.common import load_config, load_table, read_json, save_table
cfg = load_config(ROOT)

### **Figure Export Helper**


In [2]:
FIGURES = cfg['paths']['artifacts'] / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 24)
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.spines.top': False, 'axes.spines.right': False})
print('Run:', cfg['version'], '| model:', cfg['generation']['model'])
print('Outputs:', cfg['paths']['artifacts'].relative_to(ROOT))

Run: luna-v2 | model: gpt-5.6-luna
Outputs: artifacts\luna_v2


### **Load Notebook 01 and Notebook 02 Artifacts**

Build deterministic hashed requests from the frozen cohort and retrieval contexts.

In [3]:
from adaptive_rag.luna_v2.generation import (prepare_requests, batch_step, run_batches,
    generation_status, result_table)
from adaptive_rag.luna_v2.evaluation import evaluate_results
cohort = load_table(cfg, 'cohort', processed=True)
docs = load_table(cfg, 'docs', processed=True)
rankings = load_table(cfg, 'rankings')
request_plan = prepare_requests(cfg, cohort, docs, rankings)
display(request_plan.groupby(['domain', 'condition']).agg(requests=('custom_id', 'size'),
    conservative_reserve_usd=('reserved_usd', 'sum')).round(4))
print('First-pass conservative reserve: $', round(request_plan.reserved_usd.sum(), 4))

requests  conservative_reserve_usd
domain            condition                                              
natural_questions direct                    369                    0.1146
                  hybrid_rrf                369                    0.3970
                  hybrid_rrf_reranked       369                    0.3968
techqa            direct                    908                    0.3174
                  hybrid_rrf                908                    1.0584
                  hybrid_rrf_reranked       908                    1.0576

First-pass conservative reserve: $ 3.3419


---

# **Part 2: Controlled Generation Design**


### **Execution Mode and Paid-Run Controls**

The model uses reasoning none, a 256-token output cap, no tools, and explicit caching without cache breakpoints. Batch generation has no per-request synchronous latency. Additionally, keep RETRY_FAILED = False to save failed responses as unavailable, skip them, and continue the remaining requests.

In [4]:
ACTION = 'run'
RETRY_FAILED = False
REQUEST_LIMIT = 30  # used only by submit; None uses the configured chunk size

---

# **Part 3: Generate Direct, Standard RAG, and Escalated RAG Answers**


### **Resumable Batch Execution**

Completed responses are reused by request hash. Uncertain submissions stop for reconciliation. Failed responses are recorded and skipped while remaining requests continue. Run one state-machine step or the explicit multi-batch loop, then inspect saved status.

In [5]:
if ACTION == 'run':
    status = run_batches(cfg, retry_failed=RETRY_FAILED, poll_seconds=30, max_wait_hours=48)
else:
    status = batch_step(cfg, ACTION, retry_failed=RETRY_FAILED, request_limit=REQUEST_LIMIT)
display(status['counts'])
display({k: v for k, v in status.items() if k not in ('counts', 'budget_usd')})

Luna Batch processed: 100%|###########################################################| 3831/3831 [00:00<?, ?r…

,domain,condition,status,requests
0,natural_questions,direct,completed,369
1,natural_questions,hybrid_rrf,completed,369
2,natural_questions,hybrid_rrf_reranked,completed,369
3,techqa,direct,completed,896
4,techqa,direct,failed,12
5,techqa,hybrid_rrf,completed,905
6,techqa,hybrid_rrf,failed,3
7,techqa,hybrid_rrf_reranked,completed,905
8,techqa,hybrid_rrf_reranked,failed,3


{'reserved_usd_all_attempts': 3.342186,
 'active_batches': [],
 'finished': True,
 'complete': False}

---

# **Part 4: Answer Evaluation and Availability**

### **Reference-Based Evaluation**


In [6]:
results = result_table(cfg)
evaluated = evaluate_results(cfg, cohort, results)
display(evaluated.groupby(['domain', 'condition', 'status']).size().rename('requests').reset_index())
error_columns = [c for c in ['example_id', 'condition', 'status', 'error_code', 'error'] if c in results]
display(results[results.status.eq('failed')][error_columns].head(20))
status = generation_status(cfg)
print('Generation finished:', status['finished'])
print('Paired analysis examples:', evaluated.loc[evaluated.analysis_eligible, 'example_id'].nunique())

,domain,condition,status,requests
0,natural_questions,direct,completed,369
1,natural_questions,hybrid_rrf,completed,369
2,natural_questions,hybrid_rrf_reranked,completed,369
3,techqa,direct,completed,896
4,techqa,direct,failed,12
5,techqa,hybrid_rrf,completed,905
6,techqa,hybrid_rrf,failed,3
7,techqa,hybrid_rrf_reranked,completed,905
8,techqa,hybrid_rrf_reranked,failed,3


,example_id,condition,status,error_code,error
1217,techqa_DEV_Q037,hybrid_rrf_reranked,failed,unusable_response,"Incomplete, refused, empty, missing usage, or ..."
1413,techqa_DEV_Q104,direct,failed,unusable_response,"Incomplete, refused, empty, missing usage, or ..."
1443,techqa_DEV_Q114,direct,failed,unusable_response,"Incomplete, refused, empty, missing usage, or ..."
1444,techqa_DEV_Q114,hybrid_rrf,failed,unusable_response,"Incomplete, refused, empty, missing usage, or ..."
1445,techqa_DEV_Q114,hybrid_rrf_reranked,failed,unusable_response,"Incomplete, refused, empty, missing usage, or ..."
1512,techqa_DEV_Q137,direct,failed,unusable_response,"Incomplete, refused, empty, missing usage, or ..."
1572,techqa_DEV_Q157,direct,failed,unusable_response,"Incomplete, refused, empty, missing usage, or ..."
1803,techqa_DEV_Q234,direct,failed,unusable_response,"Incomplete, refused, empty, missing usage, or ..."
1912,techqa_DEV_Q270,hybrid_rrf,failed,unusable_response,"Incomplete, refused, empty, missing usage, or ..."
1913,techqa_DEV_Q270,hybrid_rrf_reranked,failed,unusable_response,"Incomplete, refused, empty, missing usage, or ..."


Generation finished: True
Paired analysis examples: 1263


### **Generation Quality and Availability Interpretation**

Generation completed for 99.5% of requests, with all 18 unavailable responses confined to 14 TechQA examples. On paired cases, Hybrid RRF improves automatic answer accuracy over direct generation by 6.5 percentage points on NQ and 8.6 points on TechQA; reranking adds only another 0.8 and 0.5 points.

RAG also reduces answer coverage, showing that some quality gain comes with more abstention. On TechQA, Hybrid RRF has slightly higher task success than reranking (14.3% versus 14.1%) despite slightly lower answer accuracy, so the extra reranking stage does not improve the overall automatic outcome. Unavailable responses remain excluded as complete paired examples rather than being scored as wrong.

---

# **Part 5: Conclusion**

Notebook 03 completed 3,813 of 3,831 Luna calls (99.5%). The 18 unavailable responses affected 14 TechQA examples, so the paired analysis retains 1,263 examples: all 369 NQ examples and 894 TechQA examples. NQ answer accuracy improved from 32.8% direct to 39.3% with Hybrid RRF and 40.1% after reranking. TechQA answer accuracy improved from 0.5% to 9.1% and 9.6%, while task success was 2.9%, 14.3%, and 14.1%; these TechQA scores remain lexical proxies pending the blind human audit. Successful calls used an estimated USD 0.631; the saved usage and reservation records make the generation cost auditable.

---